In Pandas Fundamentals, you learned to select, filter, sort, and summarize a table. Now we will turn those skills into transformations: **How do we calculate a useful variable, combine labeled tables, and express a custom rule?**

This chapter follows a practical sequence:

**Calculate → align → explore relationships → recode → apply custom logic → explain.**

By the end, you should be able to:

- Calculate new columns and row summaries using explicit input columns.
- Predict alignment by row and column labels and justify missing-value choices.
- Calculate and interpret correlations without making causal claims.
- Choose column expressions, `map()`, `replace()`, or `apply()` for a transformation.
- Write short lambda expressions and readable named functions.

No NumPy knowledge is needed. NumPy Fundamentals follows this lesson and introduces numerical arrays used by pandas and other scientific libraries.

In [ ]:
from pathlib import Path
import pandas as pd

print('Working folder:', Path.cwd().name)
print('pandas version:', pd.__version__)

The import should succeed in your verified project environment. Check the working folder before opening or rendering the activity. If needed, revisit the Reading Data setup.

## Arithmetic within a DataFrame

### Column-to-column arithmetic

The most common transformation is combining two existing columns into a new
one. Because pandas operations are **vectorized**, you write the calculation
once and it applies to every row automatically — no loop required.

In [ ]:
business_df = pd.DataFrame({
    'Product':         ['Laptop', 'Phone', 'Tablet', 'Watch', 'Headphones'],
    'Units_Sold':      [150, 300, 200, 500, 250],
    'Price_per_Unit':  [1200, 800, 600, 400, 200],
    'Cost_per_Unit':   [800, 500, 400, 250, 120],
})

business_df['Total_Revenue'] = business_df['Units_Sold'] * business_df['Price_per_Unit']
business_df['Total_Cost']    = business_df['Units_Sold'] * business_df['Cost_per_Unit']
business_df['Gross_Profit']  = business_df['Total_Revenue'] - business_df['Total_Cost']

| Operation | Symbol | Typical use |
|---|---|---|
| Addition | `+` | combining two sources, e.g. `online + retail` |
| Subtraction | `-` | profit, difference, change |
| Multiplication | `*` | totals, e.g. `price * quantity` |
| Division | `/` | rates, ratios, margins |
| Power | `**` | compound growth, e.g. `principal * (1 + rate) ** years` |
| Floor division | `//` | counting whole groups, e.g. full batches |
| Modulo | `%` | remainders, e.g. grouping IDs by their last digit |

Every one of these lines up **row by row automatically**: row 0 of
`Units_Sold` is multiplied by row 0 of `Price_per_Unit`, and so on — you never
have to say which row goes with which.

### Arithmetic with a constant (scalar)

You can also combine a column with a single number. Pandas applies it to
every row:

In [ ]:
business_df['Discounted_Price'] = business_df['Price_per_Unit'] * 0.9   # 10% off everything
business_df['Price_with_Shipping'] = business_df['Price_per_Unit'] + 50  # flat $50 shipping
business_df['Tax_Amount'] = business_df['Total_Revenue'] * 0.08          # 8% tax

### Row-wise operations (working across columns)

Summary methods like `.sum()`, `.mean()`, `.max()`, and `.min()` default to
working **down each column** — one result per column. Pass `axis=1` to work
**across each row** instead — one result per row:

In [ ]:
quarterly_sales = pd.DataFrame({
    'Q1': [100, 150, 200, 120, 180],
    'Q2': [110, 140, 220, 130, 190],
    'Q3': [120, 160, 210, 140, 200],
    'Q4': [130, 170, 230, 150, 210],
}, index=['Product_A', 'Product_B', 'Product_C', 'Product_D', 'Product_E'])

quarter_cols = ['Q1', 'Q2', 'Q3', 'Q4']
quarters = quarterly_sales[quarter_cols]
quarterly_sales['Total'] = quarters.sum(axis=1)
quarterly_sales['Average'] = quarters.mean(axis=1)
quarterly_sales['Range'] = quarters.max(axis=1) - quarters.min(axis=1)
quarterly_sales

**Select the input columns explicitly.** Otherwise, a newly added total would be included in the next average or range. Here all four quarterly values are observed; inspect missing values before interpreting a partial total.

`axis=0` (the default) collapses rows, giving one value **per column**.
`axis=1` collapses columns, giving one value **per row**. Read the axis
argument as "the direction that disappears," not "the direction you're
looking at."


## Arithmetic between DataFrames — and automatic alignment

So far we've combined columns that live in the *same* DataFrame. Pandas can
also combine **two separate DataFrames** — and this is where its most
distinctive feature shows up: **automatic alignment**.

### The clean case

When two DataFrames share the same row labels (index) and column labels,
arithmetic between them works exactly like arithmetic within one:

In [ ]:
store_a = pd.DataFrame({
    'Electronics': [50000, 55000, 60000],
    'Clothing':    [30000, 32000, 35000],
}, index=['Q1', 'Q2', 'Q3'])

store_b = pd.DataFrame({
    'Electronics': [45000, 50000, 58000],
    'Clothing':    [28000, 30000, 33000],
}, index=['Q1', 'Q2', 'Q3'])

combined_sales = store_a + store_b       # matches by label: Q1 with Q1, Electronics with Electronics
sales_difference = store_a - store_b

Notice pandas didn't need the rows or columns to be in the same *order* — it
matched Q1 with Q1 and Electronics with Electronics **by label**, not by
position. This is a key difference from a plain grid of numbers: pandas
always asks "which row/column labels match?" before it does any math.

### When the labels don't line up

If one DataFrame has a row or column the other doesn't, pandas still tries
to align everything — and fills in `NaN` (missing value) wherever a match
can't be found:

In [ ]:
df1 = pd.DataFrame({'A': [10, 20, 30], 'B': [40, 50, 60]}, index=[0, 1, 2])
df2 = pd.DataFrame({'B': [5, 10, 15, 20], 'C': [100, 200, 300, 400]}, index=[1, 2, 3, 4])

df1 + df2

```
     A     B      C
0  NaN   NaN    NaN
1  NaN  55.0    NaN
2  NaN  70.0    NaN
3  NaN   NaN    NaN
4  NaN   NaN    NaN
```

Every cell where a label exists in only one DataFrame becomes `NaN` — column
`A` only exists in `df1`, so the entire column is missing in the result.

### Taking control with `.add()`, `.sub()`, `.mul()`, `.div()`

If `NaN` isn't what you want, use the **explicit method** version of each
operator instead of the symbol. These accept a `fill_value` — the value to
use for any label that's missing from one side, *before* the operation runs:

In [ ]:
df1.add(df2, fill_value=0)   # missing entries treated as 0 before adding
df1.sub(df2, fill_value=0)   # an explicit zero assumption for missing entries

| Symbol | Explicit method | Extra control |
|---|---|---|
| `+` | `.add(other, fill_value=...)` | fill missing values before combining |
| `-` | `.sub(other, fill_value=...)` | same |
| `*` | `.mul(other, fill_value=...)` | same |
| `/` | `.div(other, fill_value=...)` | same |

**Choose a missing-value policy before filling.** Zero can make sense when an absent record means no sales; it does not mean an unknown measurement is zero. `fill_value` also fills existing missing entries on one side. A location missing on both sides remains missing. Inspect the aligned result before using it.

**Quick check:** Why is row 1, column B equal to 55, while every value in column C is missing in `df1 + df2`?

## Exploring relationships with `.corr()`

Once you have several numeric columns, a natural next question is: do any of
them move together? The correlation coefficient summarizes that relationship
as a single number between -1 and 1.

### The full correlation matrix

In [ ]:
sales_data = pd.DataFrame({
    'Advertising_Spend':      [900, 1200, 1000, 1500, 800, 1100],
    'Website_Traffic':        [4800, 5600, 5000, 6200, 4300, 5100],
    'Customer_Satisfaction':  [3.8, 4.2, 4.0, 4.5, 3.5, 4.1],
    'Sales_Revenue':          [21000, 27000, 23500, 32000, 18500, 24500],
})

correlation_matrix = sales_data.corr()   # Pearson correlation by default
correlation_matrix

`.corr()` computes the correlation between **every pair of numeric columns**
at once and returns a square table. Select numeric columns explicitly in a mixed-type table, for example `df.select_dtypes(include="number").corr()`. Missing observations are excluded separately for each pair, so pairs can use different sample sizes. For columns with variation and enough paired observations, the diagonal is 1. A constant column or insufficient data produces an undefined correlation (`NaN`), and the table is symmetric — the
correlation of A with B is the same as B with A.

| Correlation value | Interpretation |
|---|---|
| Close to +1 | strong positive relationship — both rise together |
| Close to -1 | strong negative relationship — one rises as the other falls |
| Close to 0 | little to no linear relationship |

### A single pair

If you only care about two specific columns, call `.corr()` directly on one
Series with another as the argument:

In [ ]:
sales_data['Advertising_Spend'].corr(sales_data['Sales_Revenue'])
# A strong positive linear relationship in these six invented observations.

### Finding the strongest relationships programmatically

For a wider table, scanning a whole matrix by eye gets tedious. A short loop
over the *matrix* (not the original data) can rank every pair by strength:

In [ ]:
pairs = []
cols = correlation_matrix.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):        # skip the diagonal and duplicates
        pairs.append((cols[i], cols[j], correlation_matrix.iloc[i, j]))

pairs_df = pd.DataFrame(pairs, columns=['Variable_1', 'Variable_2', 'Correlation'])
pairs_df['Strength'] = pairs_df['Correlation'].abs()
pairs_df.sort_values('Strength', ascending=False)

**Correlation is not causation**, and it only captures *linear* relationships
— two variables can be strongly related in a curved way and still show a
correlation near zero. Treat `.corr()` as a starting point for investigation,
not a final answer.


## Custom transformations: `map()`, `apply()`, and `replace()`

Arithmetic and `.corr()` cover a lot of ground, but sometimes you need logic
that isn't just "add these columns" — a lookup table, a conditional rule, or
a calculation that mixes several columns in a custom way. That's what these
three tools are for.

### `map()` — element-by-element, one Series at a time

`Series.map()` transforms every value in a Series individually, based on a
dictionary, a function, or another Series. It's the right tool for **recoding**
— turning categories into codes, or codes into labels.

In [ ]:
df = pd.DataFrame({
    'Industry': ['Tech', 'Healthcare', 'Finance', 'Tech', 'Retail'],
    'Revenue':  [100000, 150000, 80000, 120000, 90000],
})

industry_codes = {'Tech': 1, 'Healthcare': 2, 'Finance': 3, 'Retail': 4}
df['Industry_Code'] = df['Industry'].map(industry_codes)   # dictionary lookup

df['Industry_Label'] = df['Industry'].map(lambda x: f"IND_{x}")   # function

Any value not found in the dictionary becomes `NaN` — so `map()` is also a
quick way to spot categories you forgot to account for.

### `apply()` — the flexible, all-purpose tool

`DataFrame.apply()` runs a function on every **row** (`axis=1`) or every
**column** (`axis=0`); `Series.apply()` runs it on every **element**. Use it
when the calculation needs more than one column, or needs conditional logic
that doesn't fit a simple arithmetic expression.

In [ ]:
financial_df = pd.DataFrame({
    'Revenue': [1000, 1500, 800],
    'Expenses': [980, 1200, 790],
})

def cap_expenses(row):
    """Expenses can never exceed 95% of revenue."""
    max_allowed = row['Revenue'] * 0.95
    return min(row['Expenses'], max_allowed)

financial_df['Capped_Expenses'] = financial_df.apply(cap_expenses, axis=1)
financial_df

In [ ]:
scores_df = pd.DataFrame({'score': [72, 95, 80, 88]})
scores_df['category'] = scores_df['score'].apply(lambda x: 'High' if x > 80 else 'Low')
scores_df

### `replace()` — swapping specific values

`replace()` substitutes particular values (or a dictionary of many) with new
ones. Values not included in a replacement dictionary are kept unchanged; an ordinary `map()` dictionary instead produces missing values for unmatched keys.

In [ ]:
s = pd.Series(['Tech', 'Retail', 'Other'])
print(s.map({'Tech': 'Technology', 'Retail': 'Consumer Retail'}))
print(s.replace({'Tech': 'Technology', 'Retail': 'Consumer Retail'}))

df['Industry'].replace({'Tech': 'Technology', 'Retail': 'Consumer Retail'})

### Choosing between them

| Tool | Works on | Best for | Key behavior |
|---|---|---|---|
| `map()` | a Series | dictionary lookup or a value function | ordinary dictionary: unmatched keys become missing |
| `apply()` | a Series or DataFrame | custom element, row, or column functions | `axis=1` passes each row to a DataFrame function |
| `replace()` | a Series or DataFrame | replacing selected known values | unmatched values remain unchanged |

**Choose by meaning first.** Numeric column expressions and built-in methods often avoid the overhead of calling a Python function for each value or row. Dictionary mapping is a lookup, not a Python callback per value. `replace()` is not guaranteed to be faster than `map()`; performance depends on the operation, dtype, and data size. Measure equivalent calculations when speed matters.

For the expense cap above, a built-in alternative is `financial_df['Expenses'].clip(upper=financial_df['Revenue'] * 0.95)`. The row function is useful for learning `apply()`, while the built-in method expresses this particular rule directly.

## Lambda functions: inline, one-off logic

Every `apply()` example above could use a named function defined with `def`.
When the logic is short — a single expression — it's often more convenient
to write it **inline** with a **lambda function**, an anonymous, one-line
function.

```text
lambda arguments: expression
```

In [ ]:
orders = pd.DataFrame({
    'price': [100.0, 250.0, 80.0, 1200.0],
    'discount_pct': [10, 20, 0, 5],
    'total_cost': [150, 500, 700, 1200],
    'name': ['  ada ', 'BEN', ' cHEN', 'dana  '],
    'rating': [5, 4, 3, 2],
})

orders['discounted_price'] = orders.apply(
    lambda row: row['price'] * (1 - row['discount_pct'] / 100), axis=1
)

orders['segment'] = orders['total_cost'].apply(
    lambda x: 'High Value' if x > 500 else 'Medium Value' if x > 200 else 'Low Value'
)

orders['clean_name'] = orders['name'].apply(lambda x: x.strip().title())
orders

A lambda is just a shorthand — it doesn't do anything a `def` function
couldn't. Reach for one when the logic fits comfortably on one line; switch
to a named `def` function once the logic needs more than one line, gets hard
to read, or you'll reuse it elsewhere.

In [ ]:
# Getting hard to read as a lambda — better as a named function:
def classify_risk(row):
    if row['total_cost'] > 1000 and row['rating'] <= 2:
        return 'High Risk'
    elif row['total_cost'] > 500:
        return 'Medium Risk'
    return 'Low Risk'

orders['risk'] = orders.apply(classify_risk, axis=1)
orders[['total_cost', 'rating', 'risk']]

**Quick check:** would you write `orders['tax'] = orders['price'].apply(lambda x: x * 0.08)`
or `orders['tax'] = orders['price'] * 0.08`? *(The second — it's the same result with
no `apply()` overhead. Reach for `apply`/lambda only once you need logic that
a plain column expression can't express, when a named function makes a multi-branch rule easier to read. Many conditional rules can also be expressed using Boolean masks and `.loc`.)*


## Putting it together

Here's a small worked example combining several ideas from this lesson: two
stores' prices, a shopper's grocery list, and a decision about where to shop.

In [ ]:
prices = pd.DataFrame({
    'Item':   ['roll', 'bun', 'cake', 'bread'],
    'Target': [1.50, 2.00, 5.00, 16.00],
    'Kroger': [1.00, 2.50, 4.50, 17.00],
}).set_index('Item')

shopping_list = pd.DataFrame({
    'Item': ['roll', 'bun', 'cake', 'bread'],
    'Qty':  [6, 5, 3, 1],
}).set_index('Item')

# Column arithmetic + alignment: quantities times prices, matched by Item label
cost_at_target = shopping_list['Qty'] * prices['Target']
cost_at_kroger  = shopping_list['Qty'] * prices['Kroger']

totals = pd.DataFrame({
    'Target_Total': [cost_at_target.sum()],
    'Kroger_Total': [cost_at_kroger.sum()],
})

# apply() for a small piece of custom, row-level logic
totals['Cheaper_Store'] = totals.apply(
    lambda row: 'TARGET' if row['Target_Total'] < row['Kroger_Total']
                else 'KROGER' if row['Kroger_Total'] < row['Target_Total']
                else 'TIE',
    axis=1,
)
totals

Notice that the multiplication `shopping_list['Qty'] * prices['Target']`
works because both Series share the same index (`Item`) — that's the
alignment from the alignment section doing the work of matching "roll" with "roll"
automatically, even though the two tables were built separately. These invented inputs have complete prices and quantities. With real data, check for missing values before summing: the default sum skips them and could understate the bill.


## Summary cheat sheet

| Concept | Key syntax | Watch out for |
|---|---|---|
| Column-to-column arithmetic | `df['A'] + df['B']` | matches row by row automatically |
| Arithmetic with a constant | `df['A'] * 0.9` | applies to every row |
| Row-wise summary | `df[quarter_cols].sum(axis=1)` | explicitly choose input columns |
| Between-DataFrame arithmetic | `df1 + df2` | aligns by **label**, not position |
| Misaligned labels | any label missing on one side | becomes `NaN` |
| Controlled alignment | `df1.add(df2, fill_value=0)` | justify zero; both sides missing stays missing |
| Full correlation matrix | `df.corr()` | correlation ≠ causation; linear relationships only |
| Pairwise correlation | `df['A'].corr(df['B'])` | one number, -1 to 1 |
| Recode values | `series.map({...})` | unmatched values become `NaN` |
| Custom row/column logic | `df.apply(func, axis=1)` | loops under the hood — slower than column arithmetic |
| Swap known values | `series.replace({...})` | unmatched values stay unchanged |
| Inline one-off logic | `df.apply(lambda row: ..., axis=1)` | switch to `def` once it's more than one line |

## Before You Move On {#before-you-move-on}

You should be able to explain which labels a calculation matches, which columns contribute to a summary, and how unmatched values behave in a lookup. Choose a column expression when it describes the calculation clearly; use a named function when it makes a custom rule easier to understand.

Next, NumPy Fundamentals introduces arrays, positions, shapes, and broadcasting. NumPy Speedup Pandas later brings the two libraries together.

References: [pandas mapping](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html), [aligned addition](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.add.html), and [correlation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html).